# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All entities—such as record sets, fields, and columns—are referenced by their Croissant `@id` for traceability.

### Dataset Source
The dataset is defined with a Croissant schema accessible via:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
List available record sets and fields by their `@id` mappings. These identifiers will be used to access records programmatically.

In [ ]:
# Show all available record sets, their @id, and included fields/columns

record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rset in record_sets:
    print(f"RecordSet name: {rset.name}")
    print(f"  @id: {rset.id}")
    print("  Fields (by @id):")
    for field in rset.fields:
        print(f"    - {field.name}: {field.id} (dataType: {field.data_type})")
    print("")

Let's display a sample record from each record set, with all field values referenced by their `@id`.

In [ ]:
# Show a sample record for each record set, using the @id
for rset in record_sets:
    print(f'--- RecordSet: {rset.name} (@id: {rset.id}) ---')
    try:
        sample = next(dataset.records(record_set=rset.id))
        # print key, value (field @id, and value)
        for k, v in sample.items():
            print(f"{k}: {v}")
    except StopIteration:
        print('(No records available)')
    print()

## 3. Data Extraction
Load all records from the main data table(s) into DataFrames. We'll use the `@id` of each record set (see above overview) and collect them in a dictionary for further analysis.

> **Note:** The main record set for this dataset is typically named "Proteins" (@id: `cr:Proteins`), but confirm with the listing above—update the list below if more relevant record sets are present.

In [ ]:
# List the RecordSet @ids to extract (update if multiple relevant sets)
record_set_ids = [rset.id for rset in record_sets]
# Optionally, to focus on one table, e.g.:
# record_set_ids = ['cr:Proteins']

dataframes = {}
for record_set_id in record_set_ids:
    records_gen = dataset.records(record_set=record_set_id)
    records = list(records_gen)
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet @id: {record_set_id}")
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

In [ ]:
# If the main data table exists, view its columns and first 5 rows

main_record_set_id = record_set_ids[0]  # adjust if needed

if main_record_set_id in dataframes:
    print(f'Columns in table (@id: {main_record_set_id}):')
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filter by a numeric field, normalize it, and group or summarize as appropriate.

**All operations reference fields by their Croissant `@id`.**

In [ ]:
# Example: Pick a numeric field (e.g. protein molecular weight)

# Find available candidates (float or integer fields)
numeric_field_ids = []
for field in record_sets[0].fields:
    if field.data_type in ['schema:Float', 'schema:Integer', 'Float', 'Integer']:
        numeric_field_ids.append(field.id)

print('Numeric field options:', numeric_field_ids)

# Choose a numeric field for demonstration (edit as desired)
if numeric_field_ids:
    numeric_field_id = numeric_field_ids[0]  # use the first
else:
    numeric_field_id = None

record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

if numeric_field_id and numeric_field_id in df.columns:
    # Filter by threshold (for demo, e.g. value > mean)
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize
    field_norm = numeric_field_id + '_normalized'
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, field_norm]].head())

    # Try grouping by a categorical field
    group_field_id = None
    cat_candidates = [f.id for f in record_sets[0].fields if f.data_type not in ['schema:Float', 'schema:Integer', 'Float', 'Integer']]
    # Choose first non-numeric field with a small number of unique values
    for candidate in cat_candidates:
        if candidate in df.columns and df[candidate].nunique() < 20:
            group_field_id = candidate
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean','std','count'])
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical grouping field found.")
else:
    print("No numeric field found in this RecordSet.")

## 5. Visualization
Visualize distributions or relationships using fields by their `@id`. For illustration, we'll plot a histogram of the selected numeric field and, if present, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (if group_field_id chosen above)
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook we loaded a FAIR² dataset defined by Croissant, explored its structure using `@id` references, and performed basic data exploration and normalization using `mlcroissant`. Analysis and visualization steps can be extended, always referencing dataset fields via their Croissant `@id` for future-proof, standards-compliant workflows.